In [1]:
from scipy.sparse import csc_matrix

## 5. Случайный лес

In [2]:
%%time
# Задание 5. Загрузка выборок и выделение целевой переменной
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd

RANDOM_STATE = 21
DATA_DIR = Path("data")
TARGET = "IsBadBuy"

df_train = pd.read_csv(DATA_DIR / "df_train.csv")
df_valid = pd.read_csv(DATA_DIR / "df_valid.csv")

X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET]

X_valid = df_valid.drop(columns=[TARGET])
y_valid = df_valid[TARGET]

CPU times: total: 672 ms
Wall time: 2.43 s


In [3]:
%%time
# Задание 5. Кодирование категорий и заполнение пропусков
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

categorical_columns = [
    "Auction", "Make", "Model", "Trim", "SubModel", "Color",
    "Transmission", "WheelType", "Nationality", "Size",
    "TopThreeAmericanName", "BYRNO", "VNZIP1", "VNST",
]

excluded_columns = {
    "RefId", "PurchDate", "WheelTypeID",
    "PRIMEUNIT", "AUCGUART", "IsOnlineSale",
}

numeric_columns = [
    column for column in X_train.columns
    if column not in categorical_columns and column not in excluded_columns
]

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="__MISSING__"),
    OneHotEncoder(handle_unknown="ignore"),
)

preprocessor = ColumnTransformer(
    [
        ("categorical", categorical_pipeline, categorical_columns),
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
    ],
    sparse_threshold=1.0,
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)

assert X_train_encoded.shape[0] == len(y_train)
assert X_valid_encoded.shape[0] == len(y_valid)
assert X_train_encoded.shape[1] == X_valid_encoded.shape[1]

print(f"Train: {X_train_encoded.shape}, {type(X_train_encoded).__name__}")
print(f"Validation: {X_valid_encoded.shape}, {type(X_valid_encoded).__name__}")

Train: (24391, 1857), csr_matrix
Validation: (24255, 1857), csr_matrix
CPU times: total: 1.23 s
Wall time: 3.77 s


In [4]:
%%time
# Задание 5. Класс Node и неоднородность Джини
import numpy as np


class Node:
    def __init__(self, row_indices, y):
        self.row_indices = row_indices
        self.left = None
        self.right = None

        self.n_samples = len(row_indices)
        self.n_positive = int(np.count_nonzero(y[row_indices] == 1))
        self.probability = self.n_positive / self.n_samples

    def gini(self):
        p = self.probability
        return 1.0 - p ** 2 - (1.0 - p) ** 2


y_train_array = y_train.to_numpy()

CPU times: total: 0 ns
Wall time: 0 ns


In [5]:
%%time

# Задание 5. Поиск порога по критерию Джини
def best_gini_split(values, labels):
    order = np.argsort(values)
    sorted_values = values[order]
    sorted_labels = labels[order]

    cuts = np.flatnonzero(sorted_values[:-1] < sorted_values[1:])
    if cuts.size == 0:
        return None, None

    left_size = cuts + 1
    right_size = len(values) - left_size

    positive_prefix = np.cumsum(sorted_labels == 1)
    left_positive = positive_prefix[cuts]
    right_positive = positive_prefix[-1] - left_positive

    p_left = left_positive / left_size
    p_right = right_positive / right_size

    gini_left = 2 * p_left * (1 - p_left)
    gini_right = 2 * p_right * (1 - p_right)
    weighted_gini = (
                            left_size * gini_left + right_size * gini_right
                    ) / len(values)

    best = np.argmin(weighted_gini)
    cut = cuts[best]
    threshold = (sorted_values[cut] + sorted_values[cut + 1]) / 2

    return float(threshold), float(weighted_gini[best])

CPU times: total: 0 ns
Wall time: 0 ns


In [6]:
%%time

# Задание 5. Класс DecisionTreeClassifier с параметром max_features
class DecisionTreeClassifier:
    def __init__(self, max_depth=7, max_features=None, random_state=21):
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state

    def fit(self, X, y):
        X_csc = X.tocsc()
        n_features = X_csc.shape[1]
        if self.max_features is not None:
            if not isinstance(self.max_features, (int, np.integer)):
                raise TypeError("max_features должен быть целым числом или None")
            if not 1 <= self.max_features <= n_features:
                raise ValueError("max_features должен быть от 1 до числа признаков")

        self._feature_rng = np.random.default_rng(self.random_state)
        labels = np.asarray(y)
        rows = np.arange(X_csc.shape[0])
        self.root_ = self._grow(X_csc, labels, rows, depth=0)
        return self

    def _grow(self, X, y, rows, depth):
        node = Node(rows, y)
        node.feature = None
        node.threshold = None

        if depth >= self.max_depth or node.gini() == 0.0:
            return node

        feature, threshold, score = self._best_node_split(X, y, rows)

        if feature is None or node.gini() - score <= 1e-12:
            return node

        values = X[rows, feature].toarray().ravel()
        left_mask = values <= threshold

        # 2.2 	Node содержит self.left и self.right. При создании они равны None, а при разбиении _grow записывает в них дочерние узлы.

        node.feature = feature
        node.threshold = threshold
        node.left = self._grow(X, y, rows[left_mask], depth + 1)
        node.right = self._grow(X, y, rows[~left_mask], depth + 1)
        return node

    # ==================================================================================================

    def predict_proba(self, X):
        X_csc = X.tocsc()
        positive = np.empty(X_csc.shape[0], dtype=float)

        def fill_probabilities(node, rows):
            if rows.size == 0:
                return

            if node.feature is None:
                positive[rows] = node.probability
                return

            values = X_csc[rows, node.feature].toarray().ravel()
            left_mask = values <= node.threshold

            fill_probabilities(node.left, rows[left_mask])
            fill_probabilities(node.right, rows[~left_mask])

        fill_probabilities(self.root_, np.arange(X_csc.shape[0]))
        return np.column_stack((1.0 - positive, positive))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    # ==================================================================================================
    # 	2.3
    # 	Метод DecisionTreeClassifier._best_node_split() выбирает разбиение
    # 	с минимальной взвешенной неоднородностью среди всех признаков.
    # 	При построении дерева _grow вызывает этот метод через self.
    #   Эта функция находит наилучшее возможное разделение в текущем узле
    def _best_node_split(self, X, y, row_indices):
        node_labels = y[row_indices]
        best_feature = None
        best_threshold = None
        best_score = np.inf

        features = range(X.shape[1])
        if self.max_features is not None:
            features = self._feature_rng.choice(
                X.shape[1], size=self.max_features, replace=False
            )

        for feature in features:
            values = X[row_indices, feature].toarray().ravel()
            threshold, score = best_gini_split(values, node_labels)

            if score is not None and score < best_score:
                best_feature = feature
                best_threshold = threshold
                best_score = score

        return best_feature, best_threshold, best_score

CPU times: total: 0 ns
Wall time: 0 ns


In [7]:
%%time
# Задание 5. Проверка max_features и воспроизводимости дерева

from scipy.sparse import csc_matrix

X_feature_example = csc_matrix([[0, 0], [1, 1], [2, 2], [3, 3]])
y_feature_example = np.array([0, 0, 1, 1])

feature_tree = DecisionTreeClassifier(
    max_depth=1, max_features=1, random_state=21
)

feature_tree.fit(X_feature_example, y_feature_example)
first_feature = feature_tree.root_.feature
first_proba = feature_tree.predict_proba(X_feature_example)

feature_tree.fit(X_feature_example, y_feature_example)

assert first_feature in (0, 1)
assert feature_tree.root_.feature == first_feature
assert np.array_equal(feature_tree.predict_proba(X_feature_example), first_proba)
assert np.array_equal(feature_tree.predict(X_feature_example), y_feature_example)

print(f"Признак в корне: {first_feature}")
print("Разбиение и вероятности воспроизводятся.")

Признак в корне: 0
Разбиение и вероятности воспроизводятся.
CPU times: total: 15.6 ms
Wall time: 7.51 ms


In [8]:
%%time

# Задание 5. Класс RandomForestClassifier

class RandomForestClassifier:
    def __init__(self, max_depth=7, n_estimators=20, max_features=None,
                 random_state=21):
        self.max_depth = max_depth
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.random_state = random_state

    def fit(self, X, y):
        if not isinstance(self.n_estimators, (int, np.integer)):
            raise TypeError("n_estimators должен быть целым числом")
        if self.n_estimators < 1:
            raise ValueError("n_estimators должен быть не меньше 1")

        X_csr = X.tocsr()
        labels = np.asarray(y)
        n_rows, n_features = X_csr.shape
        if n_rows == 0 or n_features == 0 or labels.shape != (n_rows,):
            raise ValueError("X должен быть непустым, y должен содержать метку для каждой строки")

        max_features = self.max_features
        if max_features is None:
            max_features = max(1, int(np.sqrt(n_features)))

        seed = np.random.SeedSequence(self.random_state)
        bootstrap_seed, feature_seed = seed.spawn(2)
        bootstrap_rng = np.random.default_rng(bootstrap_seed)

        trees = []
        for tree_seed in feature_seed.spawn(self.n_estimators):
            rows = bootstrap_rng.integers(0, n_rows, size=n_rows)
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                max_features=max_features,
                random_state=tree_seed,
            )
            tree.fit(X_csr[rows], labels[rows])
            trees.append(tree)

        self.trees_ = trees
        return self

    def predict_proba(self, X):
        if not hasattr(self, "trees_"):
            raise ValueError("Сначала обучите лес через fit")

        X_csc = X.tocsc()
        probabilities = np.zeros((X_csc.shape[0], 2), dtype=float)
        for tree in self.trees_:
            probabilities += tree.predict_proba(X_csc)

        return probabilities / len(self.trees_)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

CPU times: total: 0 ns
Wall time: 0 ns


In [10]:
%%time
# Задание 5. Подготовка данных для проверки воспроизводимости

from scipy.sparse import csc_matrix

values = np.arange(20)
X_forest_example = csc_matrix(np.column_stack([values, values[::-1]]))
y_forest_example = (values >= 10).astype(int)

params = dict(max_depth=2, n_estimators=5, max_features=1, random_state=21)

CPU times: total: 0 ns
Wall time: 7.63 ms


In [11]:
%%time
# Задание 5. Проверка воспроизводимости случайного леса 1

model = RandomForestClassifier(**params).fit(X_forest_example, y_forest_example)
print(model.predict_proba(X_forest_example)[:, 1])
print(model.predict(X_forest_example))

[0.  0.  0.  0.  0.  0.  0.  0.  0.  0.2 1.  1.  1.  1.  1.  1.  1.  1.
 1.  1. ]
[0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
CPU times: total: 0 ns
Wall time: 12.9 ms


In [12]:
%%time
# Задание 5. Проверка воспроизводимости случайного леса 2

model = RandomForestClassifier(**params).fit(X_forest_example, y_forest_example)
print(model.predict_proba(X_forest_example)[:, 1])
print(model.predict(X_forest_example))

[0.  0.  0.  0.  0.  0.  0.  0.  0.  0.2 1.  1.  1.  1.  1.  1.  1.  1.
 1.  1. ]
[0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
CPU times: total: 15.6 ms
Wall time: 12.2 ms


### Теперь обучим случайный лес на train и оценим его на validation. Нужно получить Gini не ниже 0,15 и улучшить результат одиночного дерева. Наш ориентир для сравнения 0,4277 при max_depth=3.

In [13]:
%%time
# Задание 5. Обучение случайного леса и оценка Gini на валидации

from sklearn.metrics import roc_auc_score

forest = RandomForestClassifier(
    max_depth=7,
    n_estimators=20,
    max_features=None,
    random_state=21,
)
forest.fit(X_train_encoded, y_train_array)

forest_scores = forest.predict_proba(X_valid_encoded)[:, 1]
forest_auc = roc_auc_score(y_valid, forest_scores)
forest_gini = 2 * forest_auc - 1

single_tree_gini = 0.4277

print(f"ROC AUC леса на валидации: {forest_auc:.4f}")
print(f"Gini леса на валидации: {forest_gini:.4f}")
print(f"Разница с одиночным деревом: {forest_gini - single_tree_gini:+.4f}")

ROC AUC леса на валидации: 0.7201
Gini леса на валидации: 0.4403
Разница с одиночным деревом: +0.0126
CPU times: total: 16.7 s
Wall time: 25.9 s


Самописный случайный лес показал ROC AUC 0,7201 и Gini 0,4403 на валидационной выборке. Gini на 0,0126 выше результата одиночного дерева и превышает порог 0,15. Оба условия задания 5 выполнены.